# 按 ligand 浏览解析路径（按需渲染）

- 数据源：`../data/reaction_paths_all_routes.csv`
- 分组：按 `ligand_smiles` 分组；再选择 `解析树ID`，仅渲染该路径。
- 步骤：每条路径按 `步骤序号` 升序展示，每一步显示 反应物 → 产物，并标注 `反应模版`。
- 显示规则（与已有 notebook 一致）：
  - 若“和反应物砌块分子反应的中间体分子”为 NA-like（N/A/na/nan/空/None），仅显示“在zinc库里的反应物砌块分子”。
  - 否则显示“库存反应物 + 中间体反应物”。

使用说明：
- 先选择 ligand（SMILES），再选择 路径ID；只有在选择具体路径后才会渲染，不会平铺所有路径。


In [ ]:
import os, math
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, HTML
import ipywidgets as widgets

CSV_PATH = "../data/reaction_paths_all_routes.csv"
assert os.path.exists(CSV_PATH), f"CSV 文件未找到：{CSV_PATH}，请先确认路径。"

# 读取数据
usecols = [
    "ligand_smiles",
    "解析树ID",
    "步骤序号",
    "路线分数",
    "当前状态分子",
    "在zinc库里的反应物砌块分子",
    "和反应物砌块分子反应的中间体分子",
    "反应模版",
]

df = pd.read_csv(CSV_PATH, usecols=usecols)

# 统一列名简写
COL_LIG = "ligand_smiles"
COL_TREE = "解析树ID"
COL_STEP = "步骤序号"
COL_SCORE = "路线分数"
COL_PROD = "当前状态分子"
COL_STOCK = "在zinc库里的反应物砌块分子"
COL_INTER = "和反应物砌块分子反应的中间体分子"
COL_TPL = "反应模版"

# 基本信息
num_rows = len(df)
num_ligands = df[COL_LIG].nunique()
print(f"数据行数: {num_rows}, 不同 ligand 数: {num_ligands}")

# 组件
ligands = sorted(df[COL_LIG].dropna().unique().tolist())
ligand_dropdown = widgets.Dropdown(
    options=ligands,
    value=ligands[0] if ligands else None,
    description='Ligand:',
    layout=widgets.Layout(width='40%'),
)

route_dropdown = widgets.Dropdown(
    options=[("请选择路径", None)],
    value=None,
    description='路径ID:',
    layout=widgets.Layout(width='40%'),
)

header_html = widgets.HTML(value="")
output = widgets.Output()

def _is_na_like(val):
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return True
        if s.lower() in ("n/a", "na", "nan"):
            return True
    return False


def parse_list(val: str):
    if _is_na_like(val):
        return []
    if not isinstance(val, str):
        return []
    return [x.strip() for x in val.split("+") if x.strip()]


def to_svg(smiles: str, w=200, h=200):
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception:
        mol = None
    if mol is None:
        return "<div style='color:red; border:1px solid red; padding:10px;'>Invalid SMILES</div>"
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
    d.FinishDrawing()
    return d.GetDrawingText()


def render_step_html(row):
    product = str(row.get(COL_PROD, ""))
    stock_col = row.get(COL_STOCK, "")
    inter_col = row.get(COL_INTER, "")
    template = str(row.get(COL_TPL, ""))

    if _is_na_like(inter_col):
        reactants = parse_list(stock_col)
    else:
        reactants = parse_list(stock_col) + parse_list(inter_col)

    reactant_svgs = "".join([to_svg(smi) for smi in reactants])
    product_svg = to_svg(product)

    html = f"""
    <div style='margin:10px 0; padding:10px; border:1px solid #ddd; border-radius:8px;'>
      <div style='margin-bottom:8px; color:#555;'><b>反应模版</b>: {template}</div>
      <div style='display:flex; align-items:center; gap:20px;'>
        <div style='text-align:center;'>
          <div style='margin-bottom:6px; font-weight:bold;'>反应物</div>
          <div style='display:flex; gap:10px; flex-wrap:wrap;'>{reactant_svgs}</div>
        </div>
        <div style='font-size: 36px; padding: 0 12px;'>→</div>
        <div style='text-align:center;'>
          <div style='margin-bottom:6px; font-weight:bold;'>产物</div>
          {product_svg}
        </div>
      </div>
    </div>
    """
    return html


def render_tree_html(tree_df, tree_id):
    tree_df = tree_df.sort_values(by=[COL_STEP])
    steps_html = "\n".join([render_step_html(r) for _, r in tree_df.iterrows()])
    score = tree_df[COL_SCORE].dropna().unique()
    score_text = ", ".join(map(str, score)) if len(score) else "N/A"

    html = f"""
    <div style='margin:12px 0; border:1px solid #bbb; border-radius:8px; padding:10px;'>
      <div style='font-size:16px; margin-bottom:10px;'>路径 {tree_id} | 路线分数: {score_text} | 步骤数: {len(tree_df)}</div>
      {steps_html}
    </div>
    """
    return html


def render_ligand_header(lig_smi):
    sub = df[df[COL_LIG] == lig_smi]
    if sub.empty:
        return "<div style='color:red;'>该 ligand 无解析路径</div>"
    tree_ids = sub[COL_TREE].dropna().unique().tolist()
    total_steps = len(sub)
    header_svg = to_svg(lig_smi, w=180, h=180)
    return f"""
    <div>
      <div style='display:flex; align-items:center; gap:12px;'>
        <div style='font-weight:bold;'>Ligand:</div>
        {header_svg}
      </div>
      <div style='margin:6px 0; color:#444;'>共 {len(tree_ids)} 条路径，{total_steps} 个步骤。请在右侧选择具体路径以渲染。</div>
    </div>
    """


def on_ligand_change(change):
    # 更新路径下拉，不渲染路径
    lig = change["new"]
    with output:
        output.clear_output(wait=True)
    if lig is None:
        route_dropdown.options = [("请选择路径", None)]
        route_dropdown.value = None
        header_html.value = ""
        return
    sub = df[df[COL_LIG] == lig]
    tree_ids = sub[COL_TREE].dropna().unique().tolist()
    route_dropdown.options = [("请选择路径", None)] + [(tid, tid) for tid in tree_ids]
    route_dropdown.value = None
    header_html.value = render_ligand_header(lig)


def on_route_change(change):
    tree_id = change["new"]
    with output:
        output.clear_output(wait=True)
        if tree_id is None or ligand_dropdown.value is None:
            return
        sub = df[(df[COL_LIG] == ligand_dropdown.value) & (df[COL_TREE] == tree_id)]
        if sub.empty:
            display(HTML("<div style='color:red;'>该路径无数据</div>"))
            return
        html = render_tree_html(sub, tree_id)
        display(HTML(html))


ligand_dropdown.observe(on_ligand_change, names='value')
route_dropdown.observe(on_route_change, names='value')

display(HTML("<h2>按 ligand 展示解析路径</h2>"))
widgets.VBox([
    widgets.HBox([ligand_dropdown, route_dropdown]),
    header_html,
    output
])


数据行数: 992, 不同 ligand 数: 20


In [5]:
# 初始化：仅更新路径下拉，不渲染路径
on_ligand_change({"new": ligand_dropdown.value})
print("界面已准备，先选择路径以渲染")


界面已准备，先选择路径以渲染
